In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
app_types = pd.read_excel("/home/callum/Downloads/Warmwater_Tool_v2_2026-04-19.xlsx", sheet_name="Mapping_App-Typ")
param = pd.read_excel("/home/callum/Downloads/Warmwater_Tool_v2_2026-04-19.xlsx", sheet_name="Parameter_korr", skiprows=2)
df = pd.read_excel("/home/callum/Downloads/Warmwater_Tool_v2_2026-04-19.xlsx", sheet_name="Overview_korr", skiprows=3)
consumption = pd.read_excel("/home/callum/Downloads/Warmwater_Tool_v2_2026-04-19.xlsx", sheet_name="RealConsumption", skiprows=2)
pay_offset =  pd.read_excel("/home/callum/Downloads/Warmwater_Tool_v2_2026-04-19.xlsx", sheet_name="MemberPayOffSett", skiprows=2)

In [ ]:
def round_up(x):
    return np.floor(x + 0.5)

In [ ]:
def calc_topay(app_num):
    # Set apartment type and get estimated water usage
    app_type = app_types.loc[app_types["Aptm."]==app_num, 'Typ'].values[0]
    shab_col = f"Högsboet\nPre-payment\nApp Typ {app_type}\n(SEK)"
    df["Varmvatten, schablon\n=\nexpected\nconsumption\n(SEK)"] = param[shab_col]
    df["VAT / moms\non expected\nconsumption\n(SEK)"] = round_up(df["Varmvatten, schablon\n=\nexpected\nconsumption\n(SEK)"] * df["VAT / moms\n(%)"])
    df["Invoice Högsboet\nTotal monthly\npre-payment\n(SEK)"] = df["Varmvatten, schablon\n=\nexpected\nconsumption\n(SEK)"] + df["VAT / moms\non expected\nconsumption\n(SEK)"] 
    
    # lookup real consumption
    df["real\nconsumption\n(m^3)"] = consumption[app_num]
    df['calculated\nconsumption\n(SEK)'] = round_up(df["real\nconsumption\n(m^3)"] * df['calculated\ncost\nper\nm^3\n(SEK)'])
    df['calculated\ndelta\nconsumption\n(SEK)'] = df['calculated\nconsumption\n(SEK)'] - df["Varmvatten, schablon\n=\nexpected\nconsumption\n(SEK)"]
    df['calculated\nMOMS\n(SEK)'] = round_up(df['calculated\nconsumption\n(SEK)'] * df['VAT / moms\n(%)'])
    df['calculated\ndelta\nMOMS\n(SEK)'] = df['calculated\nMOMS\n(SEK)'] - df["VAT / moms\non expected\nconsumption\n(SEK)"]
    df['Invoice Högsboet\ntotal monthly\nfinal payment\n(SEK)'] = df['calculated\nconsumption\n(SEK)'] + df['calculated\nMOMS\n(SEK)']
    
    # lookup the expected consumption from member payments
    
    df['Member\nPre-payment\nExpected\nConsumption\n(SEK)'] = param[f"Member\nPre-payment\nApp Typ {app_type}\n(SEK)"]
    df['Member\nPre-payment\nVAT / moms\n(SEK)'] = round_up(df['Member\nPre-payment\nExpected\nConsumption\n(SEK)'] * df['VAT / moms\n(%)'])
    
    
    # lookup offsetting payments made
    df['Member\nPayment\nOff-setting\n(SEK)'] = pay_offset[app_num].fillna(0)
    
    # calculate offsets
    
    df['Balance \n= \nHögsboet Payment\nminus\nMember Payment'] = df['Invoice Högsboet\ntotal monthly\nfinal payment\n(SEK)'] - df['Member\nPre-payment\nExpected\nConsumption\n(SEK)'] - df['Member\nPre-payment\nVAT / moms\n(SEK)'] - df['Member\nPayment\nOff-setting\n(SEK)']
    df['Aggregated\nBalance\nat end of month'] = df['Balance \n= \nHögsboet Payment\nminus\nMember Payment'].cumsum()

    # return current balance
    final = df[['Month','Balance \n= \nHögsboet Payment\nminus\nMember Payment', 'Aggregated\nBalance\nat end of month']]
    last_row = final.dropna().iloc[-1]
    to_pay = last_row['Aggregated\nBalance\nat end of month']
    return to_pay, final

In [ ]:
for app in range(1, 32):
    topay, final = calc_topay(app)
    print(app, topay)

In [ ]:
final = final.set_index('Month', drop=True).dropna()

In [ ]:
final.plot()